# Muspelheim / Alday et al. 2017

Dataset key: `muspelheim_alday2017_public`

Source: Figshare `Muspelheim data`, DOI `10.6084/m9.figshare.3412312.v1`. The paper `10.1523/ENEURO.0311-16.2017` cites Jung et al. 2001.

The import/conversion code for this specific source is embedded directly below. It creates fixed continuous pseudo-epochs because the available BrainVision marker files contain story-level markers, not word-level trial events.

In [ ]:
import Pkg

function find_repo_root()
    candidates = unique(normpath.([
        pwd(),
        joinpath(pwd(), ".."),
        joinpath(pwd(), "..", ".."),
        joinpath(pwd(), "..", "..", ".."),
    ]))
    for candidate in candidates
        if isdir(joinpath(candidate, "notebooks")) && isdir(joinpath(candidate, "scripts"))
            return candidate
        end
    end
    error("Could not locate repository root from pwd=$(pwd()).")
end

const REPO_ROOT = find_repo_root()
const NOTEBOOK_DIR = joinpath(REPO_ROOT, "notebooks", "week_19", "data_sources")
const DATASETS_ROOT = joinpath(REPO_ROOT, "notebooks", "datasets")
const WEEK19_DOWNLOADS = joinpath(REPO_ROOT, "notebooks", "week_19", "downloads")
const PYTHON = begin
    venv_python = joinpath(REPO_ROOT, ".venv_8bit", "bin", "python")
    isfile(venv_python) ? venv_python : "python"
end

Pkg.activate(joinpath(REPO_ROOT, "notebooks", "model_test"))

using CairoMakie
using CSV
using DataFrames
using HDF5
using JSON3
using Printf
using Statistics

include(joinpath(REPO_ROOT, "notebooks", "week_15", "try_new_data_helpers.jl"))
using .Week15TryNewData

mkpath(WEEK19_DOWNLOADS)
RNG_SEED = Int(mod(time_ns(), UInt64(typemax(Int))))
println("Repo root: ", REPO_ROOT)
println("Python: ", PYTHON)
println("RNG seed: ", RNG_SEED)

In [ ]:
function bundle_files(dataset_key::AbstractString)
    dir = joinpath(DATASETS_ROOT, dataset_key)
    return (
        dir = dir,
        h5 = joinpath(dir, "epochs.hdf5"),
        events = joinpath(dir, "events.csv"),
        metadata = joinpath(dir, "metadata.json"),
    )
end

function standard_bundle_ready(dataset_key::AbstractString)
    files = bundle_files(dataset_key)
    all(isfile, [files.h5, files.events, files.metadata]) || return false
    return h5open(files.h5, "r") do f
        if haskey(f, "subjects")
            return length(keys(f["subjects"])) > 0
        end
        return true
    end
end

function maybe_run_import!(cmd::Cmd; force::Bool = false)
    if RUN_IMPORT || force
        run(cmd)
    else
        @info "RUN_IMPORT=false; not running importer. Set RUN_IMPORT=true in this notebook to rebuild/download."
    end
end
const DATASET_KEY = "muspelheim_alday2017_public"
const RUN_IMPORT = false

const MUSPELHEIM_IMPORT_PY = raw"""

from pathlib import Path
import json
import re
import urllib.request

import h5py
import mne
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "notebooks").is_dir():
    REPO_ROOT = Path("/home/benjamin/Dokumente/BA2")
DOWNLOAD_DIR = REPO_ROOT / "notebooks" / "week_19" / "downloads" / "muspelheim_figshare"
OUTPUT_ROOT = REPO_ROOT / "notebooks" / "datasets"
ARTICLE_API = "https://api.figshare.com/v2/articles/3412312"
DATASET_KEY = "muspelheim_alday2017_public"
COMPONENT = "Muspelheim naturalistic EEG"
SOURCE_COMPONENT = "https://figshare.com/articles/dataset/Muspelheim_data/3412312"
SOURCE_PAPER_DOI = "10.1523/ENEURO.0311-16.2017"
SOURCE_DATA_DOI = "10.6084/m9.figshare.3412312.v1"
USER_AGENT = "BA2-codex/1.0"
SUBJECTS = ["ali0038"]
SEGMENT_ORDER = {"full_story": 0, "aan": 1, "aav": 2, "azn": 3, "azv": 4}

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = OUTPUT_ROOT / DATASET_KEY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def fetch_json(url):
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req, timeout=60) as response:
        return json.load(response)

def subject_from_name(name):
    match = re.match(r"^(ali\d+)", name)
    return match.group(1) if match else None

def segment_from_name(name, subject):
    stem = Path(name).stem
    if stem == subject:
        return "full_story"
    return stem.removeprefix(subject + "_") or "full_story"

def download_file(file_info):
    dest = DOWNLOAD_DIR / file_info["name"]
    if dest.exists() and dest.stat().st_size == int(file_info["size"]):
        print("[skip]", dest.name)
        return dest
    print("[download]", dest.name, int(file_info["size"]) / 1_000_000, "MB")
    req = urllib.request.Request(file_info["download_url"], headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req, timeout=180) as response, dest.open("wb") as handle:
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            handle.write(chunk)
    return dest

def fixed_pseudo_epochs(raw, epoch_s=0.8, step_s=1.0, max_epochs=400):
    sfreq = float(raw.info["sfreq"])
    epoch_n = int(round(epoch_s * sfreq))
    step_n = int(round(step_s * sfreq))
    starts = np.arange(0, max(raw.n_times - epoch_n + 1, 0), step_n, dtype=int)
    starts = starts[:max_epochs]
    if len(starts) == 0:
        raise RuntimeError("Segment too short for pseudo-epochs")
    data = raw.get_data().astype(np.float32)
    epochs = np.stack([data[:, start:start + epoch_n] for start in starts], axis=2)
    times_s = np.arange(epoch_n, dtype=np.float32) / np.float32(sfreq)
    return epochs, times_s, starts.astype(np.float64) / sfreq

article = fetch_json(ARTICLE_API)
(DOWNLOAD_DIR / "article_3412312.json").write_text(json.dumps(article, indent=2), encoding="utf-8")
(DOWNLOAD_DIR / "files_manifest.json").write_text(json.dumps(article["files"], indent=2), encoding="utf-8")
for file_info in article["files"]:
    if subject_from_name(file_info["name"]) in set(SUBJECTS):
        download_file(file_info)

all_events = []
subject_trial_counts = []
selected_subjects = []
first_channel_names = None
with h5py.File(OUTPUT_DIR / "epochs.hdf5", "w") as h5:
    h5.attrs["dataset_key"] = DATASET_KEY
    h5.attrs["component"] = COMPONENT
    h5.attrs["source_component"] = SOURCE_COMPONENT
    h5.attrs["source_data_doi"] = SOURCE_DATA_DOI
    h5.attrs["source_paper_doi"] = SOURCE_PAPER_DOI
    h5.attrs["epoching_note"] = "fixed continuous pseudo-epochs; not stimulus locked"
    subjects_group = h5.create_group("subjects")
    for subject_id, subject in enumerate(SUBJECTS, start=1):
        epoch_parts = []
        event_parts = []
        times_ref = None
        channel_names_ref = None
        sfreq_ref = None
        epoch_offset = 0
        segment_offset_s = 0.0
        for vhdr_path in sorted(DOWNLOAD_DIR.glob(f"{subject}*.vhdr")):
            segment = segment_from_name(vhdr_path.name, subject)
            print("[load]", vhdr_path.name, "segment=", segment)
            raw = mne.io.read_raw_brainvision(vhdr_path, preload=True, verbose="ERROR")
            raw.set_eeg_reference("average", projection=False, verbose="ERROR")
            raw.filter(l_freq=0.1, h_freq=30.0, verbose="ERROR")
            raw.resample(250.0, verbose="ERROR")
            epochs, times_s, onsets_s = fixed_pseudo_epochs(raw)
            if times_ref is None:
                times_ref = times_s
                channel_names_ref = list(raw.ch_names)
                sfreq_ref = float(raw.info["sfreq"])
            if channel_names_ref != list(raw.ch_names):
                raise RuntimeError("Channel mismatch between segments")
            n_trials = epochs.shape[2]
            event_parts.append(pd.DataFrame({
                "dataset_key": DATASET_KEY,
                "component": COMPONENT,
                "subject_id": subject_id,
                "subject_label": subject,
                "session_label": "story",
                "run_label": segment,
                "segment": segment,
                "segment_order": SEGMENT_ORDER.get(segment, 99),
                "pseudo_onset_s": onsets_s,
                "continuous_order_s": segment_offset_s + onsets_s,
                "epoch_index": np.arange(epoch_offset + 1, epoch_offset + n_trials + 1, dtype=int),
                "source_file": vhdr_path.name,
            }))
            epoch_parts.append(epochs)
            epoch_offset += n_trials
            segment_offset_s += raw.n_times / float(raw.info["sfreq"])
        subject_epochs = np.concatenate(epoch_parts, axis=2)
        subject_events = pd.concat(event_parts, ignore_index=True)
        all_events.extend(subject_events.to_dict(orient="records"))
        selected_subjects.append(subject)
        subject_trial_counts.append({"subject_label": subject, "n_trials": int(subject_epochs.shape[2])})
        first_channel_names = first_channel_names or channel_names_ref
        group = subjects_group.create_group(subject)
        group.create_dataset("epochs", data=subject_epochs, compression="gzip", compression_opts=4)
        group.create_dataset("times_s", data=times_ref.astype(np.float32))
        group.create_dataset("channel_names", data=np.asarray(channel_names_ref, dtype=h5py.string_dtype(encoding="utf-8")))
        group.attrs["subject_id"] = subject_id
        group.attrs["subject_label"] = subject
        group.attrs["sfreq_hz"] = float(sfreq_ref)
        group.attrs["n_channels"] = int(subject_epochs.shape[0])
        group.attrs["n_timepoints"] = int(subject_epochs.shape[1])
        group.attrs["n_trials"] = int(subject_epochs.shape[2])
        group.attrs["source_set_relpath"] = str(DOWNLOAD_DIR)
        group.attrs["source_eventlist_relpath"] = "fixed continuous pseudo-epochs generated in muspelheim_alday2017_public.ipynb"

events_df = pd.DataFrame(all_events).sort_values(["subject_id", "epoch_index"])
events_df.to_csv(OUTPUT_DIR / "events.csv", index=False)
metadata = {
    "dataset_key": DATASET_KEY,
    "component": COMPONENT,
    "source_component": SOURCE_COMPONENT,
    "source_data_doi": SOURCE_DATA_DOI,
    "source_paper_doi": SOURCE_PAPER_DOI,
    "source_root_listing": ARTICLE_API,
    "source_processing_scripts": "notebooks/week_19/data_sources/muspelheim_alday2017_public.ipynb",
    "reader_docs": "https://mne.tools/stable/generated/mne.io.read_raw_brainvision.html",
    "selected_subjects": selected_subjects,
    "preferred_channels": [ch for ch in ["Pz", "CPz", "POz", "Cz", "Fz", "FCz"] if first_channel_names and ch in first_channel_names],
    "recommended_sort_columns": ["continuous_order_s", "pseudo_onset_s", "segment", "segment_order", "epoch_index"],
    "hdf5_path": "epochs.hdf5",
    "events_csv_path": "events.csv",
    "notes": [
        "Alday et al. 2017 cites Jung et al. 2001 and links this Figshare dataset.",
        "The Figshare BrainVision marker files available here contain story-level markers, not word-level events.",
        "This bundle uses fixed continuous pseudo-epochs for exploratory ERP-image-like shape inspection only.",
    ],
    "official_source_examples": {
        "figshare_manifest": "notebooks/week_19/downloads/muspelheim_figshare/files_manifest.json",
        "figshare_article": "notebooks/week_19/downloads/muspelheim_figshare/article_3412312.json",
    },
    "subject_trial_counts": subject_trial_counts,
    "sort_columns_present": list(events_df.columns),
}
(OUTPUT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
(OUTPUT_DIR / "README.md").write_text("# Muspelheim Naturalistic EEG\n\nConverted in notebooks/week_19/data_sources/muspelheim_alday2017_public.ipynb.\n", encoding="utf-8")
print("[done]", OUTPUT_DIR)

"""

if standard_bundle_ready(DATASET_KEY)
    println("Bundle already ready: ", bundle_files(DATASET_KEY).dir)
elseif RUN_IMPORT
    run(Cmd([PYTHON, "-c", MUSPELHEIM_IMPORT_PY]))
else
    @info "RUN_IMPORT=false; not running Muspelheim importer. Set RUN_IMPORT=true to download/rebuild."
end

In [ ]:
TARGET_SIZE = nothing
N_SAMPLES_PER_SORT = 16
N_COLS = 4

if standard_bundle_ready(DATASET_KEY)
    bundle = load_clean_dataset_bundle(DATASET_KEY)
    display(external_dataset_summary_df([bundle]))
    display(available_sort_columns_df([bundle]))

    sort_columns = available_sort_columns(bundle)
    sort_audit = sort_order_audit_df(bundle; sort_columns = sort_columns, include_merged = true)
    display(sort_audit)
    @assert all(sort_audit.status .== "ok") "Sort-order audit failed for $(DATASET_KEY)."

    plot_all_dataset_sort_previews(bundle;
        sort_columns = sort_columns,
        n_samples = N_SAMPLES_PER_SORT,
        target_size = TARGET_SIZE,
        rng_seed = RNG_SEED,
        n_cols = N_COLS,
        merge_subjects = true,
    )
else
    files = bundle_files(DATASET_KEY)
    println("Bundle unavailable/incomplete: ", files.dir)
    if isdefined(Main, :SOURCE_NOTE)
        println("Source note: ", SOURCE_NOTE)
    end
    println("No preview generated for this notebook until a complete bundle exists.")
end
